In [1]:
# =============================================================================
# STEP 0: FETCH HISTORICAL DATA  &  TRAIN BASE MODEL
#
# Fetches the latest 3 years of violent crime data from the Chicago SODA API
# (ijzp-q8t2), engineers features matching the training pipeline, and trains
# the XGBoost model. Data is cached locally as parquet after first download.
#
# ★ Run this cell once to build/rebuild the base model.
# ★ STEP 2/3 can later pull 2026 live data and retrain incrementally.
# =============================================================================

import os, json, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# ── Install / import dependencies ────────────────────────────────────────────
import subprocess, sys
for pkg in ["requests", "h3", "xgboost", "scikit-learn"]:
    try:
        __import__(pkg.replace("-", "_").split("[")[0])
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--quiet"])

import requests, h3
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)

DEPLOY_DIR = "../deploy_Render/deployment"
API_HIST       = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"  # 2001–present
HIST_CACHE     = os.path.join(DEPLOY_DIR, "_historical_cache.parquet")
CACHE_META     = os.path.join(DEPLOY_DIR, "_cache_meta.json")
H3_RES         = 8                           # ★ Resolution 8 — matches training notebook
N_YEARS        = 3                           # Fetch N years back from today's date

# ★ Violent crime types — exact match to training notebook
VIOLENT_TYPES = ["BATTERY", "ASSAULT", "ROBBERY"]

EPSILON = 1e-6

# ═════════════════════════════════════════════════════════════════════════════
# 1. FETCH HISTORICAL DATA FROM SODA API
# ═════════════════════════════════════════════════════════════════════════════
def _paginated_fetch(api_url, where_clause, label="", batch_size=50000):
    """Generic paginated SODA API fetcher."""
    all_rows, offset = [], 0
    while True:
        params = {
            "$where" : where_clause,
            "$limit" : batch_size,
            "$offset": offset,
            "$order" : "date ASC",
        }
        resp = requests.get(api_url, params=params, timeout=120)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        all_rows.extend(batch)
        offset += len(batch)
        print(f"  {label}… {offset:,} rows")
        if len(batch) < batch_size:
            break
    return all_rows


def fetch_historical_data():
    """
    Maintain a rolling N_YEARS window of violent crime data.
    Tracks start_date and end_date in a metadata file.
    On each run: fetch only new days, trim expired old days.
    First run downloads the full window.
    Returns (DataFrame, has_new_data: bool).
    """
    os.makedirs(DEPLOY_DIR, exist_ok=True)

    today       = datetime.now().date()
    window_start = today.replace(year=today.year - N_YEARS)

    # ── Load cache metadata ───────────────────────────────────────────────
    meta = None
    if os.path.exists(CACHE_META):
        with open(CACHE_META) as f:
            meta = json.load(f)

    # ── If cache exists and is current, do incremental update ─────────────
    if meta and os.path.exists(HIST_CACHE):
        cached_end   = datetime.strptime(meta["end_date"], "%Y-%m-%d").date()
        cached_start = datetime.strptime(meta["start_date"], "%Y-%m-%d").date()

        if cached_end >= today:
            # Already up to date
            df = pd.read_parquet(HIST_CACHE)
            print(f"✓ Cache is current: {len(df):,} rows "
                  f"({cached_start} → {cached_end})")
            return df, False

        # Fetch only the delta: from day after cached_end to today
        fetch_from = cached_end + timedelta(days=1)
        print(f"Fetching new data: {fetch_from} → {today} "
              f"(cache had up to {cached_end}) …")

        where = (f"date >= '{fetch_from.isoformat()}' "
                 f"AND date <= '{today.isoformat()}T23:59:59' "
                 f"AND primary_type in('BATTERY','ASSAULT','ROBBERY')")
        new_rows = _paginated_fetch(API_HIST, where, label="delta")
        print(f"  ✓ Fetched {len(new_rows):,} new records")

        # Load existing cache
        df = pd.read_parquet(HIST_CACHE)

        # Append new data
        if new_rows:
            new_df = pd.DataFrame(new_rows)
            df = pd.concat([df, new_df], ignore_index=True)

        # Trim old data outside the rolling window
        df["_date"] = pd.to_datetime(df["date"], errors="coerce")
        before_trim = len(df)
        df = df[df["_date"].dt.date >= window_start].copy()
        df = df.drop(columns=["_date"])
        trimmed = before_trim - len(df)
        if trimmed > 0:
            print(f"  Trimmed {trimmed:,} records older than {window_start}")

        # Save updated cache and metadata
        df.to_parquet(HIST_CACHE, index=False)
        new_meta = {
            "start_date": str(window_start),
            "end_date"  : str(today),
            "total_rows": len(df),
            "updated_at": str(datetime.now()),
        }
        with open(CACHE_META, "w") as f:
            json.dump(new_meta, f, indent=2)

        print(f"✓ Cache updated: {len(df):,} rows ({window_start} → {today})")
        return df, len(new_rows) > 0

    # ── First run: full download ──────────────────────────────────────────
    print(f"First run — fetching {N_YEARS} years ({window_start} → {today}) …")

    all_rows = []
    for yr in range(window_start.year, today.year + 1):
        yr_start = max(window_start, datetime(yr, 1, 1).date()).isoformat()
        yr_end   = min(today, datetime(yr, 12, 31).date()).isoformat()

        where = (f"date >= '{yr_start}' AND date <= '{yr_end}T23:59:59' "
                 f"AND primary_type in('BATTERY','ASSAULT','ROBBERY')")
        rows = _paginated_fetch(API_HIST, where, label=f"{yr}")
        all_rows.extend(rows)
        print(f"  ✓ {yr}: {len(rows):,} records")

    df = pd.DataFrame(all_rows)
    df.to_parquet(HIST_CACHE, index=False)

    meta = {
        "start_date": str(window_start),
        "end_date"  : str(today),
        "total_rows": len(df),
        "updated_at": str(datetime.now()),
    }
    with open(CACHE_META, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"✓ Cache created: {len(df):,} rows ({window_start} → {today})")
    return df, True


# ═════════════════════════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING — exact replica of training notebook
# ═════════════════════════════════════════════════════════════════════════════

def engineer_features(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Transform raw API rows into the labelled, feature-rich DataFrame
    that EXACTLY matches the original training pipeline.
    """
    df = raw.copy()

    # ── 2a. Basic cleaning ────────────────────────────────────────────────
    df["Date"]      = pd.to_datetime(df["date"], errors="coerce")
    df["latitude"]  = pd.to_numeric(df["latitude"],  errors="coerce")
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["primary_type"] = df["primary_type"].astype(str).str.upper().str.strip()
    df = df.dropna(subset=["Date", "latitude", "longitude"])

    # ── 2b. Filter to violent crimes only ─────────────────────────────────
    rows_before = len(df)
    df = df[df["primary_type"].isin(VIOLENT_TYPES)].copy()
    print(f"  Filtered to violent crimes: {rows_before:,} → {len(df):,}")

    # ── 2c. Chicago boundary filter ───────────────────────────────────────
    chicago_mask = (
        (df["latitude"] > 41.6) & (df["latitude"] < 42.1) &
        (df["longitude"] > -88.0) & (df["longitude"] < -87.5)
    )
    df = df[chicago_mask].copy()

    # ── 2d. H3 tile assignment (resolution 8) ─────────────────────────────
    df["h3_address"] = df.apply(
        lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RES), axis=1
    )

    # ── 2e. Shift assignment — exact match to training notebook ───────────
    # np.select with hour.between(6,13) / hour.between(14,21) / default overnight
    hour = df["Date"].dt.hour
    df["shift"] = np.select(
        [hour.between(6, 13), hour.between(14, 21)],
        ["morning_noon", "afternoon_night"],
        default="overnight"
    )
    # shift_date: anchor overnight pre-6am crimes to previous calendar day
    df["shift_date"] = df["Date"].dt.floor("D")
    df.loc[hour < 6, "shift_date"] -= pd.Timedelta(days=1)

    # ── 2f. Build master grid: tile × date × shift (zero-fill) ────────────
    tile_shift_counts = (
        df.groupby(["h3_address", "shift_date", "shift"])
        .size()
        .reset_index(name="crime_count")
    )
    shift_order = ["morning_noon", "afternoon_night", "overnight"]
    idx = pd.MultiIndex.from_product(
        [df["h3_address"].unique(),
         pd.date_range(df["shift_date"].min(), df["shift_date"].max(), freq="D"),
         shift_order],
        names=["h3_address", "shift_date", "shift"]
    )
    master_grid = pd.DataFrame(index=idx).reset_index()
    final_df = pd.merge(
        master_grid, tile_shift_counts,
        on=["h3_address", "shift_date", "shift"], how="left"
    )
    final_df["crime_count"] = final_df["crime_count"].fillna(0)
    final_df["target"] = (final_df["crime_count"] > 0).astype(int)

    # Add Date column (shift_date + shift start hour)
    shift_start_hour = {"morning_noon": 6, "afternoon_night": 14, "overnight": 22}
    final_df["Date"] = (
        final_df["shift_date"]
        + pd.to_timedelta(final_df["shift"].map(shift_start_hour), unit="h")
    )

    n_tiles = final_df["h3_address"].nunique()
    print(f"  Master grid: {len(final_df):,} rows × {n_tiles:,} tiles")

    # ── 2g. lag_1d: same-shift 1-day lag ──────────────────────────────────
    final_df = final_df.sort_values(["h3_address", "shift", "shift_date"]).reset_index(drop=True)
    final_df["lag_1d"] = (
        final_df.groupby(["h3_address", "shift"])["crime_count"].shift(1)
    )

    # ── 2h. Rolling averages (grouped by tile AND shift) ──────────────────
    final_df["rolling_7d_mean"] = (
        final_df.groupby(["h3_address", "shift"])["crime_count"]
        .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
    )
    final_df["rolling_30d_mean"] = (
        final_df.groupby(["h3_address", "shift"])["crime_count"]
        .transform(lambda x: x.shift(1).rolling(window=30, min_periods=1).mean())
    )

    # ── 2i. Tile Crime Density Percentile (EWMA-based) ────────────────────
    daily_tile = (
        final_df.groupby(["h3_address", "shift_date"])["crime_count"]
        .sum().reset_index()
        .sort_values(["h3_address", "shift_date"])
        .reset_index(drop=True)
    )
    daily_tile["tile_ewma_crime"] = (
        daily_tile.groupby("h3_address")["crime_count"]
        .transform(lambda x: x.shift(1).ewm(halflife=30, min_periods=1).mean())
    )
    daily_tile["tile_crime_density_percentile"] = (
        daily_tile.groupby("shift_date")["tile_ewma_crime"]
        .transform(lambda x: x.rank(pct=True))
        .fillna(0.5)
    )

    # ── 2j. Tile Momentum (fast EWMA / slow EWMA) ────────────────────────
    daily_tile["tile_ewma_fast"] = (
        daily_tile.groupby("h3_address")["crime_count"]
        .transform(lambda x: x.shift(1).ewm(halflife=7, min_periods=1).mean())
    )
    daily_tile["tile_ewma_slow"] = (
        daily_tile.groupby("h3_address")["crime_count"]
        .transform(lambda x: x.shift(1).ewm(halflife=30, min_periods=1).mean())
    )
    daily_tile["tile_momentum"] = (
        daily_tile["tile_ewma_fast"] / (daily_tile["tile_ewma_slow"] + EPSILON)
    ).fillna(1.0)

    # Merge daily features onto final_df
    final_df = final_df.merge(
        daily_tile[["h3_address", "shift_date",
                    "tile_crime_density_percentile", "tile_momentum"]],
        on=["h3_address", "shift_date"], how="left"
    )

    # ── 2k. Cyclical time features — exact formulas from training ─────────
    final_df["day_of_week"] = final_df["Date"].dt.dayofweek
    final_df["month"]       = final_df["Date"].dt.month

    final_df["day_sin"]   = np.sin(2 * np.pi * final_df["day_of_week"] / 7)
    final_df["day_cos"]   = np.cos(2 * np.pi * final_df["day_of_week"] / 7)
    # ★ (month - 1) / 12  — matches training notebook exactly
    final_df["month_sin"] = np.sin(2 * np.pi * (final_df["month"] - 1) / 12)
    final_df["month_cos"] = np.cos(2 * np.pi * (final_df["month"] - 1) / 12)

    # Shift dummies
    final_df["is_afternoon_night"] = (final_df["shift"] == "afternoon_night").astype(int)
    final_df["is_overnight"]       = (final_df["shift"] == "overnight").astype(int)

    # ── 2l. Spatial lag: neighbor_lag_1d ──────────────────────────────────
    print("  Computing spatial neighbour lag (this may take a minute) …")
    final_df_sorted = final_df.sort_values(["h3_address", "Date"]).reset_index(drop=True)
    prev_crime = (
        final_df_sorted.groupby("h3_address")["crime_count"]
        .shift(1).fillna(0)
    )
    prev_lookup = dict(zip(
        zip(final_df_sorted["Date"], final_df_sorted["h3_address"]),
        prev_crime
    ))

    def calc_spatial_lag(row):
        neighbors = h3.grid_disk(row["h3_address"], k=1)
        neighbors = [n for n in neighbors if n != row["h3_address"]]
        return sum(prev_lookup.get((row["Date"], n), 0) for n in neighbors)

    final_df["neighbor_lag_1d"] = final_df.apply(calc_spatial_lag, axis=1)

    # ── 2m. City-normalised rolling features ──────────────────────────────
    city_daily_mean = (
        final_df.groupby("shift_date")["crime_count"]
        .mean().reset_index()
        .rename(columns={"crime_count": "city_daily_mean"})
        .sort_values("shift_date")
    )
    city_daily_mean["city_baseline"] = (
        city_daily_mean["city_daily_mean"]
        .shift(1).expanding().mean()
    )
    city_daily_mean["city_baseline"] = city_daily_mean["city_baseline"].fillna(
        city_daily_mean["city_daily_mean"].iloc[0]
    )
    final_df = final_df.merge(
        city_daily_mean[["shift_date", "city_baseline"]],
        on="shift_date", how="left"
    )

    final_df["rolling_30d_mean_norm"] = (
        final_df["rolling_30d_mean"] / (final_df["city_baseline"] + EPSILON)
    )
    final_df["rolling_7d_mean_norm"] = (
        final_df["rolling_7d_mean"] / (final_df["city_baseline"] + EPSILON)
    )
    final_df["neighbor_lag_1d_norm"] = (
        final_df["neighbor_lag_1d"] / (final_df["city_baseline"] + EPSILON)
    )

    # Drop rows with NaN in feature columns — matches training notebook
    # (NaN occurs in first-day cold-start rows where lag/rolling can't be computed)
    _before = len(final_df)
    final_df = final_df.dropna(subset=[
        "lag_1d", "rolling_7d_mean_norm", "rolling_30d_mean_norm",
        "tile_crime_density_percentile", "tile_momentum",
        "neighbor_lag_1d_norm", "target"
    ]).copy()
    print(f"  Dropped {_before - len(final_df):,} cold-start rows with NaN")
    print(f"  ✓ Feature engineering complete — {len(final_df):,} rows")
    return final_df, daily_tile


# ═════════════════════════════════════════════════════════════════════════════
# 3. RETRAIN THE XGBOOST MODEL — same pipeline as training notebook
# ═════════════════════════════════════════════════════════════════════════════

# ★ Exact feature columns from training notebook
CATEGORICAL_COLS = ["is_afternoon_night", "is_overnight"]
NUMERIC_COLS = [
    "lag_1d",
    "rolling_7d_mean_norm",
    "rolling_30d_mean_norm",
    "tile_crime_density_percentile",
    "tile_momentum",
    "day_sin", "day_cos",
    "month_sin", "month_cos",
    "neighbor_lag_1d_norm",
]
FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS


def retrain_model(final_df: pd.DataFrame, daily_tile: pd.DataFrame,
                  deploy_dir=DEPLOY_DIR):
    """
    Train a calibrated XGBoost with the same ColumnTransformer pipeline
    as the original training notebook.
    """
    # Drop rows with NaN in features or target
    df = final_df.dropna(subset=FEATURE_COLS + ["target"]).copy()

    X = df[FEATURE_COLS]
    y = df["target"]

    print(f"\nTraining data: {len(df):,} tile-shift-days "
          f"across {df['h3_address'].nunique():,} tiles")
    print(f"  Positive rate (violent crime): {y.mean():.3%}")

    # ── Time-based train/test split (80/20 by date) ───────────────────────
    split_date = df["shift_date"].quantile(0.8)
    train_mask = df["shift_date"] <= split_date
    test_mask  = ~train_mask

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]
    print(f"  Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")

    # ── ColumnTransformer — matches training notebook ─────────────────────
    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OrdinalEncoder(
                handle_unknown="use_encoded_value", unknown_value=-1
            ), CATEGORICAL_COLS),
            ("num", "passthrough", NUMERIC_COLS),
        ],
        remainder="drop"
    )

    # ── XGBoost base model ────────────────────────────────────────────────
    base_xgb = XGBClassifier(
        n_estimators     = 300,
        max_depth        = 5,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1),
        eval_metric      = "logloss",
        random_state     = 42,
        use_label_encoder= False,
    )

    # ── Assemble pipeline: preprocessor → XGBoost ─────────────────────────
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", base_xgb),
    ])

    # ── Calibrate with CalibratedClassifierCV ─────────────────────────────
    calibrated = CalibratedClassifierCV(pipe, cv=3, method="sigmoid")
    calibrated.fit(X_train, y_train)

    # ── Evaluate ──────────────────────────────────────────────────────────
    y_prob = calibrated.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob) if y_test.nunique() > 1 else 0.0
    print(f"\n✓ ROC-AUC on hold-out: {auc:.4f}")

    # Find threshold targeting ~15-20% precision with reasonable recall
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    viable = np.where(precision[:-1] >= 0.12)[0]
    if len(viable) > 0:
        best_idx  = viable[np.argmax(recall[viable])]
        threshold = float(thresholds[best_idx])
    else:
        threshold = 0.15
    # Precision & recall at the selected threshold
    y_pred_at_thresh = (y_prob >= threshold).astype(int)
    tp = int(((y_pred_at_thresh == 1) & (y_test == 1)).sum())
    fp = int(((y_pred_at_thresh == 1) & (y_test == 0)).sum())
    fn = int(((y_pred_at_thresh == 0) & (y_test == 1)).sum())
    prec_at_thresh = round(tp / max(tp + fp, 1), 4)
    rec_at_thresh  = round(tp / max(tp + fn, 1), 4)
    print(f"  Selected threshold: {threshold:.3f}")
    print(f"  Precision @ threshold: {prec_at_thresh:.4f}")
    print(f"  Recall    @ threshold: {rec_at_thresh:.4f}")

    # ── Save deployment artefacts ─────────────────────────────────────────
    os.makedirs(deploy_dir, exist_ok=True)

    # 1) Calibrated pipeline
    joblib.dump(calibrated, os.path.join(deploy_dir, "xgb_calibrated_pipeline.joblib"))

    # 2) Tile baseline — exact columns from training notebook deployment cell
    last_date = final_df["shift_date"].max()
    tile_baseline = (
        final_df[final_df["shift_date"] == last_date]
        .groupby("h3_address")
        .agg(
            rolling_30d_mean_norm         = ("rolling_30d_mean_norm", "mean"),
            rolling_7d_mean_norm          = ("rolling_7d_mean_norm", "mean"),
            tile_crime_density_percentile = ("tile_crime_density_percentile", "mean"),
            tile_momentum                 = ("tile_momentum", "mean"),
            neighbor_lag_1d_norm          = ("neighbor_lag_1d_norm", "mean"),
            lag_1d                        = ("lag_1d", "mean"),
        )
        .reset_index()
    )
    tile_baseline.to_csv(os.path.join(deploy_dir, "tile_baseline.csv"), index=False)

    # 3) Metadata JSON
    base_rate = float(y.mean())
    meta = {
        "model"             : "XGBoost",
        "roc_auc"           : round(auc, 4),
        "precision"         : prec_at_thresh,
        "recall"            : rec_at_thresh,
        "threshold"         : round(threshold, 4),
        "base_rate"         : round(base_rate, 5),
        "feature_cols"      : FEATURE_COLS,
        "categorical_cols"  : CATEGORICAL_COLS,
        "numeric_cols"      : NUMERIC_COLS,
        "total_tiles"       : int(tile_baseline["h3_address"].nunique()),
        "n_train_rows"      : int(len(X_train)),
        "n_test_rows"       : int(len(X_test)),
        "trained_on"        : f"{(datetime.now().date().replace(year=datetime.now().year - N_YEARS))} to {datetime.now().date()}",
        "trained_at"        : str(datetime.now()),
        "data_source"       : API_HIST,
        "shift_hours"       : {
            "morning_noon"   : "06:00-13:59",
            "afternoon_night": "14:00-21:59",
            "overnight"      : "22:00-05:59",
        },
    }
    with open(os.path.join(deploy_dir, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\n✓ Deployment artefacts saved to '{deploy_dir}/':")
    print(f"  • xgb_calibrated_pipeline.joblib")
    print(f"  • tile_baseline.csv  ({len(tile_baseline):,} tiles)")
    print(f"  • metadata.json")
    print(f"\n→ Now run STEP 1 to load the refreshed model.\n")

    return calibrated, meta


# ═════════════════════════════════════════════════════════════════════════════
# 4. RUN IT
# ═════════════════════════════════════════════════════════════════════════════
raw_df, _ = fetch_historical_data()
final_df, daily_tile = engineer_features(raw_df)
model, metadata = retrain_model(final_df, daily_tile)


Fetching new data: 2026-03-18 → 2026-03-18 (cache had up to 2026-03-17) …
  ✓ Fetched 0 new records
  Trimmed 152 records older than 2023-03-18
✓ Cache updated: 223,234 rows (2023-03-18 → 2026-03-18)

⏭  No new data — skipping retrain.
   → Run STEP 1 to load the model.



In [2]:
# =============================================================================
# STEP 1: INFERENCE ENGINE
# Load packaged model and predict crime probability for any tile-shift
#
# ★ Updated features:
#   - live_override_from_api(): fetch today's crimes and compute fresh lag_1d
#   - Beat/district mapping via H3 centroid → police beat spatial join
#   - Adjustable threshold support in predict()
# =============================================================================

import joblib
import json
import numpy as np
import pandas as pd
import requests
import h3
from datetime import datetime, date, timedelta

API_URL = "https://data.cityofchicago.org/resource/f6bk-yv3r.json"
H3_RES  = 8
VIOLENT_TYPES = ["BATTERY", "ASSAULT", "ROBBERY"]

class CrimePredictionEngine:
    """
    Loads the packaged XGBoost deployment artefacts and scores
    any tile x date x shift combination.
    """

    SHIFT_MAP = {
        "morning_noon"   : {"is_afternoon_night": 0, "is_overnight": 0, "hour_start": 6},
        "afternoon_night": {"is_afternoon_night": 1, "is_overnight": 0, "hour_start": 14},
        "overnight"      : {"is_afternoon_night": 0, "is_overnight": 1, "hour_start": 22},
    }

    def __init__(self, deploy_dir="deployment"):
        print("Loading deployment package...")
        self.pipeline  = joblib.load(f"{deploy_dir}/xgb_calibrated_pipeline.joblib")
        self.baselines = pd.read_csv(f"{deploy_dir}/tile_baseline.csv")
        with open(f"{deploy_dir}/metadata.json") as f:
            self.meta = json.load(f)

        self.threshold    = self.meta["threshold"]
        self.feature_cols = self.meta["feature_cols"]
        print(f"✓ Model loaded  — ROC-AUC {self.meta['roc_auc']} | "
              f"Threshold {self.threshold}")
        print(f"✓ {len(self.baselines):,} tiles available for scoring")

    # ── Live data override ────────────────────────────────────────────────────
    def live_override_from_api(self, target_date=None):
        """
        Fetch recent violent crimes from the SODA API and compute
        fresh lag_1d counts per H3 tile for the given date.
        Returns a DataFrame with [h3_address, lag_1d] suitable
        for passing to predict(override_tiles=...).
        """
        if target_date is None:
            target_date = date.today()
        if isinstance(target_date, str):
            target_date = pd.Timestamp(target_date).date()

        # Fetch the previous 2 days of data to compute lag
        start = target_date - timedelta(days=2)
        end   = target_date

        print(f"  Fetching live data {start} → {end} from API …")
        params = {
            "$where": (f"date >= '{start.isoformat()}' "
                       f"AND date < '{(end + timedelta(days=1)).isoformat()}' "
                       f"AND primary_type in('BATTERY','ASSAULT','ROBBERY')"),
            "$limit": 50000,
            "$order": "date ASC",
        }
        resp = requests.get(API_URL, params=params, timeout=60)
        resp.raise_for_status()
        rows = resp.json()

        if not rows:
            print("  ⚠ No recent violent crimes returned from API.")
            return None

        df = pd.DataFrame(rows)
        df["Date"]      = pd.to_datetime(df["date"], errors="coerce")
        df["latitude"]  = pd.to_numeric(df.get("latitude"),  errors="coerce")
        df["longitude"] = pd.to_numeric(df.get("longitude"), errors="coerce")
        df = df.dropna(subset=["Date", "latitude", "longitude"])

        df["h3_address"] = df.apply(
            lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RES), axis=1
        )

        # Compute crimes per tile on the day BEFORE target_date (= lag_1d)
        yesterday = target_date - timedelta(days=1)
        yest_df = df[df["Date"].dt.date == yesterday]
        lag_counts = (
            yest_df.groupby("h3_address").size()
            .reset_index(name="lag_1d")
        )

        print(f"  ✓ Live override: {len(lag_counts):,} tiles with "
              f"fresh lag_1d from {yesterday}")
        return lag_counts

    # ── Feature construction ──────────────────────────────────────────────────
    def _build_features(self, query_date, shift,
                        override_tiles=None) -> pd.DataFrame:
        if shift not in self.SHIFT_MAP:
            raise ValueError(f"shift must be one of: {list(self.SHIFT_MAP.keys())}")

        tiles = self.baselines.copy()

        if override_tiles is not None:
            tiles = tiles.merge(
                override_tiles[['h3_address', 'lag_1d']].rename(
                    columns={'lag_1d': 'lag_1d_fresh'}),
                on='h3_address', how='left'
            )
            tiles['lag_1d'] = tiles['lag_1d_fresh'].fillna(tiles['lag_1d'])
            tiles.drop(columns=['lag_1d_fresh'], inplace=True)

        s = self.SHIFT_MAP[shift]
        tiles['is_afternoon_night'] = s['is_afternoon_night']
        tiles['is_overnight']       = s['is_overnight']

        dt  = pd.Timestamp(query_date)
        dow = dt.dayofweek
        mon = dt.month

        tiles['day_sin']   = np.sin(2 * np.pi * dow / 7)
        tiles['day_cos']   = np.cos(2 * np.pi * dow / 7)
        tiles['month_sin'] = np.sin(2 * np.pi * (mon - 1) / 12)
        tiles['month_cos'] = np.cos(2 * np.pi * (mon - 1) / 12)

        return tiles

    # ── Predict ───────────────────────────────────────────────────────────────
    def predict(self, query_date, shift, top_n=20,
                override_tiles=None, return_all=False,
                threshold_override=None) -> pd.DataFrame:
        """
        Score all tiles. Accepts optional threshold_override to use
        instead of the model's default threshold.
        """
        if isinstance(query_date, str):
            query_date = pd.Timestamp(query_date).date()

        threshold = threshold_override if threshold_override is not None else self.threshold

        tiles = self._build_features(query_date, shift, override_tiles)
        X     = tiles[self.feature_cols]

        probs   = self.pipeline.predict_proba(X)[:, 1]
        flagged = (probs >= threshold).astype(int)

        results = tiles[['h3_address']].copy()
        results['crime_probability'] = probs.round(4)
        results['flagged']           = flagged
        results['risk_tier'] = pd.cut(
            probs,
            bins  = [0, 0.10, 0.20, 0.35, 1.0],
            labels= ["Low", "Moderate", "High", "Critical"]
        )
        results['shift']      = shift
        results['query_date'] = str(query_date)
        results = results.sort_values('crime_probability', ascending=False)

        if return_all:
            return results.reset_index(drop=True)

        flagged_df = results[results['flagged'] == 1]
        if len(flagged_df) < top_n:
            return results.head(top_n).reset_index(drop=True)
        return flagged_df.head(top_n).reset_index(drop=True)

    # ── Summary ───────────────────────────────────────────────────────────────
    def summarise(self, results, threshold_used=None):
        n_flagged  = results['flagged'].sum()
        n_total    = len(self.baselines)
        n_critical = (results['risk_tier'] == 'Critical').sum()
        n_high     = (results['risk_tier'] == 'High').sum()
        t = threshold_used or self.threshold

        print("\n" + "="*55)
        print(f"PATROL DISPATCH SUMMARY")
        print(f"Date  : {results['query_date'].iloc[0]}")
        print(f"Shift : {results['shift'].iloc[0]}")
        print("="*55)
        print(f"Tiles flagged for patrol : {n_flagged:>5,} / {n_total:,} tiles")
        print(f"  Critical risk (>35%)   : {n_critical:>5,}")
        print(f"  High risk    (20-35%)  : {n_high:>5,}")
        base_rate = self.meta.get('base_rate', 0.067)
        precision_est = self.meta.get('precision', 0.197)
        recall_est    = self.meta.get('recall', 0.437)
        print(f"Dispatch threshold       : {t:.3f}")
        print(f"Expected crimes caught   : ~{int(n_flagged * precision_est * recall_est):,} "
              f"of ~{int(n_total * 3 * base_rate):,} "
              f"daily city-wide")
        print("="*55)


# ── Helper: retrain model with 2026 live data ────────────────────────────────
def retrain_with_live_data(engine_instance):
    """
    Fetch 2026 data from f6bk-yv3r, check for delta, and if new data exists,
    combine with cached historical data and retrain the model.
    Returns (engine, was_retrained).
    """
    import os, json as _json
    from datetime import datetime as _dt

    API_2026    = "https://data.cityofchicago.org/resource/f6bk-yv3r.json"
    DEPLOY_DIR  = "deployment"
    DELTA_CACHE = os.path.join(DEPLOY_DIR, "_2026_delta_cache.json")
    HIST_CACHE  = os.path.join(DEPLOY_DIR, "_historical_cache.parquet")

    # Check previous 2026 row count
    last_count = 0
    if os.path.exists(DELTA_CACHE):
        with open(DELTA_CACHE) as _f:
            last_count = _json.load(_f).get("total_rows", 0)

    # Fetch 2026 violent crimes
    print("Fetching 2026 data from f6bk-yv3r …")
    all_rows, offset = [], 0
    while True:
        params = {
            "$where": "year='2026' AND primary_type in('BATTERY','ASSAULT','ROBBERY')",
            "$limit": 50000, "$offset": offset, "$order": "date ASC",
        }
        resp = requests.get(API_2026, params=params, timeout=120)
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        all_rows.extend(batch)
        offset += len(batch)
        if len(batch) < 50000:
            break

    current_df = pd.DataFrame(all_rows)
    delta = len(current_df) - last_count
    print(f"  2026 records: {len(current_df):,}  (Δ {delta:+,} new)")

    # Save new count
    os.makedirs(DEPLOY_DIR, exist_ok=True)
    with open(DELTA_CACHE, "w") as _f:
        _json.dump({"total_rows": len(current_df), "fetched_at": str(_dt.now())}, _f)

    if delta == 0:
        print("  ⏭  No new 2026 data — model unchanged.")
        return engine_instance, False

    # Load historical cache and combine
    if not os.path.exists(HIST_CACHE):
        print("  ⚠ No historical cache found. Run STEP 0 first.")
        return engine_instance, False

    hist_df = pd.read_parquet(HIST_CACHE)
    combined = pd.concat([hist_df, current_df], ignore_index=True)
    print(f"  Combined: {len(combined):,} rows "
          f"(historical {len(hist_df):,} + 2026 {len(current_df):,})")

    # Import STEP 0 functions (they're in global scope from running STEP 0)
    print("  Retraining model …")
    final_df, daily_tile = engineer_features(combined)
    model, meta = retrain_model(final_df, daily_tile)

    # Reload engine
    new_engine = CrimePredictionEngine(DEPLOY_DIR)
    return new_engine, True


# ── Instantiate once ─────────────────────────────────────────────────────────
engine = CrimePredictionEngine("deployment")


Loading deployment package...
✓ Model loaded  — ROC-AUC 0.7785 | Threshold 0.0552
✓ 849 tiles available for scoring


In [ ]:
# =============================================================================
# STEP 2: INTERACTIVE USER INPUT — predict which tiles have crime
#
# ★ New controls:
#   🔄 Refresh Live Data  — fetches today's crimes from API for fresh lag_1d
#   🎚  Threshold slider   — adjust dispatch sensitivity
#   🏷  Risk tier filter   — show only selected tiers in the table/chart
#   🏢 District filter    — focus on a specific police district
# =============================================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
from shapely.geometry import shape
import h3 as h3lib

# ── Build beat/district lookup (tile → beat, district) ────────────────────────
print("Building tile → beat/district lookup …")

# Load beat boundaries
_beats_url = "https://data.cityofchicago.org/resource/n9it-hstw.json?$limit=5000"
_beats_raw = pd.read_json(_beats_url)
if "the_geom" in _beats_raw.columns:
    beats_gdf = gpd.GeoDataFrame(
        _beats_raw.drop(columns=["the_geom"]).copy(),
        geometry=_beats_raw["the_geom"].apply(
            lambda g: shape(g) if isinstance(g, dict) else None),
        crs="EPSG:4326"
    )
else:
    beats_gdf = gpd.GeoDataFrame(_beats_raw.copy(), geometry=None, crs="EPSG:4326")
if "geometry" in beats_gdf.columns:
    beats_gdf = beats_gdf[beats_gdf.geometry.notna()].copy()

# Spatial join: map every model tile to a beat/district
_tile_h3 = engine.baselines[["h3_address"]].drop_duplicates().copy()
_tile_h3[["_lat", "_lon"]] = _tile_h3["h3_address"].apply(
    lambda x: pd.Series(h3lib.cell_to_latlng(x)))
_tile_pts = gpd.GeoDataFrame(
    _tile_h3, geometry=gpd.points_from_xy(_tile_h3["_lon"], _tile_h3["_lat"]),
    crs="EPSG:4326")

_beat_col = "beat_num" if "beat_num" in beats_gdf.columns else (
    "beat" if "beat" in beats_gdf.columns else None)
_dist_col = "district" if "district" in beats_gdf.columns else None
_join_cols = ["geometry"] + ([_beat_col] if _beat_col else []) + ([_dist_col] if _dist_col else [])

_joined = gpd.sjoin(_tile_pts, beats_gdf[_join_cols], how="left", predicate="within")

tile_beat_map = {}  # h3_address -> (beat, district)
for _, r in _joined.iterrows():
    b = str(r.get(_beat_col, "Unknown")) if _beat_col and pd.notna(r.get(_beat_col)) else "Unknown"
    d = str(r.get(_dist_col, ""))        if _dist_col and pd.notna(r.get(_dist_col)) else ""
    tile_beat_map[r["h3_address"]] = (b, d)

# Build sorted district list for the dropdown
_all_districts = sorted({d for _, d in tile_beat_map.values() if d and d != ""})
_all_beats     = sorted({b for b, _ in tile_beat_map.values() if b != "Unknown"})

print(f"✓ Mapped {len(tile_beat_map):,} tiles → {len(_all_districts)} districts, "
      f"{len(_all_beats)} beats")

# ── Shared state for live override ────────────────────────────────────────────
_live_override_cache = {"df": None, "label": "Not loaded"}

# ── UI widgets ────────────────────────────────────────────────────────────────
date_picker = widgets.DatePicker(
    description = "Date:",
    value       = pd.Timestamp.today().date(),
    layout      = widgets.Layout(width="280px")
)

shift_dropdown = widgets.Dropdown(
    options = [
        ("Morning / Noon  (06:00–13:59)", "morning_noon"),
        ("Afternoon / Night (14:00–21:59)", "afternoon_night"),
        ("Overnight  (22:00–05:59)", "overnight"),
    ],
    value       = "afternoon_night",
    description = "Shift:",
    layout      = widgets.Layout(width="380px")
)

top_n_slider = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description="Top N tiles:",
    layout=widgets.Layout(width="380px")
)

# ★ NEW: Threshold slider
threshold_slider = widgets.FloatSlider(
    value  = engine.threshold,
    min    = 0.05,
    max    = 0.50,
    step   = 0.01,
    description = "Threshold:",
    readout_format = ".2f",
    layout = widgets.Layout(width="380px"),
    style  = {"description_width": "90px"},
)

# ★ NEW: Risk tier filter
tier_filter = widgets.SelectMultiple(
    options     = ["Critical", "High", "Moderate", "Low"],
    value       = ["Critical", "High", "Moderate", "Low"],
    description = "Show tiers:",
    layout      = widgets.Layout(width="280px", height="100px"),
    style       = {"description_width": "90px"},
)

# ★ NEW: District filter
district_dropdown = widgets.Dropdown(
    options     = [("All districts", "ALL")] + [(f"District {d}", d) for d in _all_districts],
    value       = "ALL",
    description = "District:",
    layout      = widgets.Layout(width="280px"),
    style       = {"description_width": "90px"},
)

# ★ NEW: Beat filter (dynamic, updated when district changes)
beat_dropdown = widgets.Dropdown(
    options     = [("All beats", "ALL")] + [(f"Beat {b}", b) for b in _all_beats],
    value       = "ALL",
    description = "Beat:",
    layout      = widgets.Layout(width="280px"),
    style       = {"description_width": "90px"},
)

def _update_beats_on_district_change(change):
    """When district changes, filter the beat dropdown to only show beats in that district."""
    dist = change["new"]
    if dist == "ALL":
        opts = [("All beats", "ALL")] + [(f"Beat {b}", b) for b in _all_beats]
    else:
        beats_in_dist = sorted({
            b for b, d in tile_beat_map.values()
            if d == dist and b != "Unknown"
        })
        opts = [("All beats in district", "ALL")] + [(f"Beat {b}", b) for b in beats_in_dist]
    beat_dropdown.options = opts
    beat_dropdown.value   = "ALL"

district_dropdown.observe(_update_beats_on_district_change, names="value")

# ★ NEW: Live data refresh button
live_btn = widgets.Button(
    description  = "🔄 Fetch 2026 & Retrain",
    button_style = "warning",
    layout       = widgets.Layout(width="200px", height="36px"),
)
live_status = widgets.Label(value="Live data: not loaded")

def on_live_refresh(b):
    global engine
    live_status.value = "Fetching 2026 data & checking for updates …"
    try:
        # 1) Retrain if there's new 2026 data
        engine, was_retrained = retrain_with_live_data(engine)
        if was_retrained:
            live_status.value = "✓ Model retrained with 2026 data"
        else:
            live_status.value = "✓ No new data — model unchanged"

        # 2) Also fetch fresh lag overrides for the selected date
        qdate = date_picker.value or pd.Timestamp.today().date()
        override_df = engine.live_override_from_api(target_date=qdate)
        _live_override_cache["df"] = override_df
        n = len(override_df) if override_df is not None else 0
        tag = "retrained + " if was_retrained else ""
        _live_override_cache["label"] = f"✓ {tag}{n} tiles with live lag ({qdate})"
        live_status.value = _live_override_cache["label"]
    except Exception as ex:
        live_status.value = f"✗ Error: {ex}"

live_btn.on_click(on_live_refresh)

# Main run button
run_button = widgets.Button(
    description  = "▶  Run Prediction",
    button_style = "primary",
    layout       = widgets.Layout(width="200px", height="36px")
)

out = widgets.Output()

# ── Risk tier colour map ──────────────────────────────────────────────────────
TIER_COLORS = {
    "Critical": "\033[91m",
    "High"    : "\033[93m",
    "Moderate": "\033[94m",
    "Low"     : "\033[92m",
}
RESET = "\033[0m"

# ── Helper: filter results by district/beat/tier ──────────────────────────────
def filter_results(results, district, beat, tiers):
    """Apply district, beat, and tier filters to results DataFrame."""
    df = results.copy()

    # Add beat/district columns
    df["_beat"]     = df["h3_address"].map(lambda h: tile_beat_map.get(h, ("Unknown",""))[0])
    df["_district"] = df["h3_address"].map(lambda h: tile_beat_map.get(h, ("Unknown",""))[1])

    if district != "ALL":
        df = df[df["_district"] == district]
    if beat != "ALL":
        df = df[df["_beat"] == beat]
    if tiers:
        df = df[df["risk_tier"].isin(tiers)]

    return df

# ── On-click handler ──────────────────────────────────────────────────────────
def on_run(b):
    with out:
        clear_output(wait=True)

        qdate     = date_picker.value
        shift     = shift_dropdown.value
        top_n     = top_n_slider.value
        threshold = threshold_slider.value
        tiers     = list(tier_filter.value)
        district  = district_dropdown.value
        beat      = beat_dropdown.value

        if qdate is None:
            print("Please select a date.")
            return

        # Use live override if loaded
        override = _live_override_cache["df"]
        override_tag = ""
        if override is not None:
            override_tag = "  [🔄 using live lag data]"

        print(f"Scoring {len(engine.baselines):,} tiles for "
              f"{qdate}  |  shift: {shift}  |  threshold: {threshold:.2f}"
              f"{override_tag}")

        # Score ALL tiles first (for city-wide summary)
        all_results = engine.predict(
            query_date         = qdate,
            shift              = shift,
            override_tiles     = override,
            return_all         = True,
            threshold_override = threshold,
        )

        # Apply filters for display
        display_results = filter_results(all_results, district, beat, tiers)

        # Operational summary
        engine.summarise(all_results, threshold_used=threshold)

        if district != "ALL" or beat != "ALL":
            area_label = f"District {district}" if beat == "ALL" else f"Beat {beat}"
            print(f"\n📍 Filtered to: {area_label}  "
                  f"({len(display_results):,} tiles shown)")

        # ── Top N table ───────────────────────────────────────────────────
        top_df = display_results.head(top_n)
        if len(top_df) == 0:
            print("\nNo tiles match the selected filters.")
            return

        print(f"\nTop {min(top_n, len(top_df))} Highest-Risk Tiles:")
        print(f"{'Rank':<5} {'H3 Tile':<20} {'Beat':<8} {'Dist':<6} "
              f"{'Probability':>12} {'Risk Tier':<12} {'Flag'}")
        print("-" * 78)

        for rank, (_, row) in enumerate(top_df.iterrows(), 1):
            tier = str(row['risk_tier'])
            flag = "🚨 DISPATCH" if row['flagged'] else "   monitor"
            color = TIER_COLORS.get(tier, "")
            b, d  = tile_beat_map.get(row['h3_address'], ('?','?'))
            print(f"{rank:<5} {row['h3_address']:<20} {b:<8} {d:<6} "
                  f"{row['crime_probability']:>12.4f} "
                  f"{color}{tier:<12}{RESET} {flag}")

        # ── Charts ────────────────────────────────────────────────────────
        tier_palette = {
            "Critical": "#C0392B", "High": "#E67E22",
            "Moderate": "#2980B9", "Low" : "#27AE60"
        }

        fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(top_df) * 0.35)))
        fig.subplots_adjust(wspace=0.45, left=0.28, right=0.97,
                            top=0.90, bottom=0.08)

        # Panel 1: Crime probability per tile (filtered)
        bar_colors = [tier_palette.get(str(t), "#95A5A6")
                      for t in top_df['risk_tier']]
        axes[0].barh(
            top_df['h3_address'][::-1],
            top_df['crime_probability'][::-1],
            color=list(reversed(bar_colors)),
            alpha=0.85, edgecolor='white', height=0.7
        )
        axes[0].axvline(threshold, color='black', linestyle='--',
                        linewidth=1.5, label=f"Threshold ({threshold:.2f})")
        area_str = ""
        if district != "ALL":
            area_str = f"  |  District {district}"
        if beat != "ALL":
            area_str = f"  |  Beat {beat}"
        axes[0].set_xlabel("Crime Probability (calibrated)", fontsize=10)
        axes[0].set_title(
            f"Top {len(top_df)} Highest-Risk Tiles\n"
            f"{qdate}  |  {shift.replace('_',' ').title()}{area_str}",
            fontweight='bold', fontsize=11)
        axes[0].legend(fontsize=8)
        axes[0].tick_params(axis='y', labelsize=7.5)
        axes[0].grid(axis='x', alpha=0.3)
        axes[0].set_xlim(0, max(top_df['crime_probability'].max() * 1.2, 0.3))

        # Panel 2: Risk tier distribution (city-wide or filtered area)
        tier_counts = display_results['risk_tier'].value_counts().reindex(
            ["Critical", "High", "Moderate", "Low"], fill_value=0)
        tier_bar_colors = [tier_palette[t] for t in tier_counts.index]
        bars = axes[1].bar(
            tier_counts.index, tier_counts.values,
            color=tier_bar_colors, alpha=0.85, edgecolor='white', width=0.6)
        for bar, val in zip(bars, tier_counts.values):
            if len(display_results) > 0:
                pct = val / len(display_results) * 100
                axes[1].text(bar.get_x() + bar.get_width()/2,
                             bar.get_height() + len(display_results)*0.005,
                             f"{val:,}\n({pct:.1f}%)",
                             ha='center', fontsize=9, fontweight='bold')
        axes[1].set_ylim(0, max(tier_counts.max() * 1.25, 1))
        axes[1].set_ylabel("Number of Tiles", fontsize=10)
        axes[1].set_title(
            f"Risk Tier Distribution — {len(display_results):,} Tiles\n"
            f"{qdate}  |  {shift.replace('_',' ').title()}{area_str}",
            fontweight='bold', fontsize=11)
        axes[1].grid(axis='y', alpha=0.3)
        legend_patches = [mpatches.Patch(color=tier_palette[t], label=t)
                          for t in ["Critical","High","Moderate","Low"]]
        axes[1].legend(handles=legend_patches, fontsize=8, loc='upper right')
        plt.show()

        # ── CSV export ────────────────────────────────────────────────────
        export_df = display_results[display_results['flagged'] == 1]
        export_path = f"patrol_briefing_{qdate}_{shift}.csv"
        export_df.to_csv(export_path, index=False)
        print(f"\n✓ Flagged tiles exported to: {export_path}")

run_button.on_click(on_run)

# ── Render UI ─────────────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════╗")
print("║     VIOLENT CRIME PREDICTION — PATROL DISPATCH              ║")
print("║     XGBoost  |  Adjustable Threshold  |  Live Data Feed     ║")
print("╚══════════════════════════════════════════════════════════════╝\n")

display(
    widgets.VBox([
        widgets.HBox([date_picker, shift_dropdown]),
        widgets.HBox([top_n_slider, threshold_slider]),
        widgets.HBox([district_dropdown, beat_dropdown, tier_filter]),
        widgets.HBox([live_btn, live_status]),
        run_button,
        out
    ])
)


Building tile → beat/district lookup …
✓ Mapped 849 tiles → 23 districts, 257 beats
╔══════════════════════════════════════════════════════════════╗
║     VIOLENT CRIME PREDICTION — PATROL DISPATCH              ║
║     XGBoost  |  Adjustable Threshold  |  Live Data Feed     ║
╚══════════════════════════════════════════════════════════════╝



In [ ]:
# =============================================================================
# STEP 3: MAP VISUALISATION — Flagged Tiles + Beat Overlay
#
# ★ New controls: threshold slider, district/beat filter, live data,
#   risk tier filter — all applied to the map as well
#
# Requires: folium
#   Inputs from previous cells: engine, beats_gdf, tile_beat_map,
#   _live_override_cache
# =============================================================================

import sys, subprocess
try:
    import folium
    import folium.plugins as plugins
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "branca", "--quiet"])
    import folium
    import folium.plugins as plugins
import h3
import json
import numpy as np
import pandas as pd
from IPython.display import display
import ipywidgets as widgets
from IPython.display import clear_output, HTML

# ── Load Community Area boundaries ─────────────────────────────────────────────
import geopandas as gpd

if 'community_gdf' not in globals() or community_gdf is None:
    print("Loading Chicago Community Area boundaries …")
    _comm_geojson_url = "https://data.cityofchicago.org/resource/igwz-8jzy.geojson"
    try:
        community_gdf = gpd.read_file(_comm_geojson_url)
        community_gdf = community_gdf[community_gdf.geometry.notna()].copy()

        # Normalise column names
        _col_map = {}
        for _orig, _target in [
            ("COMMUNITY", "community"), ("name", "community"),
            ("AREA_NUMBE", "area_numbe"), ("area_num_1", "area_numbe"),
            ("area_num", "area_numbe"),
        ]:
            if _orig in community_gdf.columns and _target not in community_gdf.columns:
                _col_map[_orig] = _target
        if _col_map:
            community_gdf = community_gdf.rename(columns=_col_map)

        print(f"  ✓ Loaded {len(community_gdf):,} community areas")
    except Exception as _e:
        print(f"  ⚠ Could not load community areas ({_e})")
        community_gdf = None

# ── Helpers (same as before) ──────────────────────────────────────────────────
def prob_to_hex(prob, threshold=0.150):
    if prob < threshold:
        return "#A8D5A2"
    t = min((prob - threshold) / (0.40 - threshold), 1.0)
    r = int(231 * t + 39  * (1 - t))
    g = int(76  * t + 174 * (1 - t))
    b = int(60  * t + 96  * (1 - t))
    return f"#{r:02X}{g:02X}{b:02X}"

# ── Core map builder (updated to accept threshold + filters) ──────────────────
def build_patrol_map(
    results, beats_gdf, threshold=0.150,
    show_all_tiles=False, map_zoom=11,
    district_filter="ALL", beat_filter="ALL", tier_filter_list=None,
    community_gdf=None,
):
    m = folium.Map(
        location=[41.8781, -87.6298], zoom_start=map_zoom,
        tiles="CartoDB positron")

    # ── Layer 1: Beat boundaries ──────────────────────────────────────────
    beats_layer = folium.FeatureGroup(name="Police Beats", show=True)
    beats_4326 = beats_gdf.to_crs("EPSG:4326")

    _bcol = "beat_num" if "beat_num" in beats_4326.columns else (
        "beat" if "beat" in beats_4326.columns else None)
    _dcol = "district" if "district" in beats_4326.columns else None

    for _, beat_row in beats_4326.iterrows():
        beat_num = beat_row.get(_bcol, "Unknown") if _bcol else "Unknown"
        district = beat_row.get(_dcol, "")         if _dcol else ""

        # Optionally highlight the selected district/beat
        is_selected = (
            (district_filter == "ALL" or str(district) == district_filter) and
            (beat_filter == "ALL" or str(beat_num) == beat_filter)
        )
        style = {
            "fillColor"   : "#1A237E" if is_selected and district_filter != "ALL" else "transparent",
            "color"       : "#1A237E",
            "weight"      : 2.5 if is_selected and district_filter != "ALL" else 1.4,
            "fillOpacity" : 0.06 if is_selected and district_filter != "ALL" else 0,
            "dashArray"   : "" if is_selected and district_filter != "ALL" else "4 4",
        }

        folium.GeoJson(
            data=beat_row.geometry.__geo_interface__,
            style_function=lambda feat, s=style: s,
            tooltip=folium.Tooltip(
                f"<b>Beat {beat_num}</b><br>District {district}", sticky=False)
        ).add_to(beats_layer)

    beats_layer.add_to(m)

    # ── Layer 1b: Community Area boundaries ───────────────────────────────
    if community_gdf is not None and len(community_gdf) > 0:
        comm_layer = folium.FeatureGroup(name="Community Areas", show=False)
        comm_4326  = community_gdf.to_crs("EPSG:4326")

        for _, crow in comm_4326.iterrows():
            area_name = crow.get("community", "Unknown")
            area_num  = crow.get("area_numbe", "")
            geom      = crow.geometry
            if geom is None:
                continue

            folium.GeoJson(
                data=geom.__geo_interface__,
                style_function=lambda feat: {
                    "fillColor"   : "transparent",
                    "color"       : "#6A1B9A",
                    "weight"      : 2.0,
                    "fillOpacity" : 0,
                    "dashArray"   : "6 3",
                },
                tooltip=folium.Tooltip(
                    f"<b>{str(area_name).title()}</b><br>"
                    f"Community Area {area_num}",
                    sticky=False)
            ).add_to(comm_layer)

        comm_layer.add_to(m)

    # ── Apply district/beat/tier filters to results ───────────────────────
    display_df = results.copy()
    display_df["_beat"]     = display_df["h3_address"].map(
        lambda h: tile_beat_map.get(h, ("Unknown",""))[0])
    display_df["_district"] = display_df["h3_address"].map(
        lambda h: tile_beat_map.get(h, ("Unknown",""))[1])

    if district_filter != "ALL":
        display_df = display_df[display_df["_district"] == district_filter]
    if beat_filter != "ALL":
        display_df = display_df[display_df["_beat"] == beat_filter]
    if tier_filter_list:
        display_df = display_df[display_df["risk_tier"].isin(tier_filter_list)]

    # ── Layer 2: Flagged tiles ────────────────────────────────────────────
    flagged_layer = folium.FeatureGroup(name="🚨 Flagged Tiles (Dispatch)", show=True)
    flagged_df    = display_df[display_df["flagged"] == 1].copy()

    for _, row in flagged_df.iterrows():
        prob     = float(row["crime_probability"])
        tier     = str(row["risk_tier"])
        h3_addr  = row["h3_address"]
        colour   = prob_to_hex(prob, threshold)
        boundary = h3.cell_to_boundary(h3_addr)

        beat_num = row.get("_beat", "Unknown")
        district = row.get("_district", "")

        tooltip_html = (
            f"<div style='font-family:Arial;font-size:13px;'>"
            f"<b>🚨 DISPATCH RECOMMENDED</b><br>"
            f"<b>Tile:</b> {h3_addr}<br>"
            f"<b>Police beat:</b> {beat_num}"
            + (f" (District {district})" if district else "") + "<br>"
            f"<b>Risk tier:</b> {tier}<br>"
            f"<b>Crime probability:</b> {prob:.1%}<br>"
            f"<b>Date / Shift:</b> {row['query_date']} | "
            f"{row['shift'].replace('_',' ').title()}"
            f"</div>")

        folium.Polygon(
            locations=[(lat_, lon_) for lat_, lon_ in boundary],
            color=colour, weight=1.5, fill=True,
            fill_color=colour, fill_opacity=0.72,
            tooltip=folium.Tooltip(tooltip_html, sticky=True),
            popup=folium.Popup(tooltip_html, max_width=280)
        ).add_to(flagged_layer)

    flagged_layer.add_to(m)

    # ── Layer 3: Monitor tiles ────────────────────────────────────────────
    monitor_layer = folium.FeatureGroup(name="⚠️ Monitor Tiles", show=show_all_tiles)
    monitor_df = display_df[
        (display_df["flagged"] == 0) &
        (display_df["crime_probability"] >= threshold * 0.65)
    ].copy()

    for _, row in monitor_df.iterrows():
        prob     = float(row["crime_probability"])
        h3_addr  = row["h3_address"]
        boundary = h3.cell_to_boundary(h3_addr)
        beat_num = row.get("_beat", "Unknown")
        district = row.get("_district", "")

        tooltip_html = (
            f"<div style='font-family:Arial;font-size:12px;'>"
            f"<b>⚠️ Monitor</b><br>"
            f"Tile: {h3_addr}<br>"
            f"Police beat: {beat_num}"
            + (f" (District {district})" if district else "") + "<br>"
            f"Probability: {prob:.1%}<br>"
            f"(Below dispatch threshold of {threshold:.0%})"
            f"</div>")

        folium.Polygon(
            locations=[(lat_, lon_) for lat_, lon_ in boundary],
            color="#F39C12", weight=1.0, fill=True,
            fill_color="#F39C12", fill_opacity=0.30,
            tooltip=folium.Tooltip(tooltip_html, sticky=True),
        ).add_to(monitor_layer)

    monitor_layer.add_to(m)

    folium.LayerControl(position="topright", collapsed=False).add_to(m)
    plugins.MiniMap(tile_layer="CartoDB positron",
                    zoom_level_offset=-6, width=180, height=180).add_to(m)

    # ── Legend ─────────────────────────────────────────────────────────────
    query_date = results["query_date"].iloc[0]
    shift_name = results["shift"].iloc[0].replace("_", " ").title()
    n_flagged  = int(flagged_df["flagged"].sum()) if len(flagged_df) > 0 else 0
    n_total    = len(display_df)
    n_critical = int((flagged_df["risk_tier"] == "Critical").sum()) if len(flagged_df) > 0 else 0
    n_high     = int((flagged_df["risk_tier"] == "High").sum()) if len(flagged_df) > 0 else 0
    n_monitor  = len(monitor_df)

    area_note = ""
    if district_filter != "ALL":
        area_note = f"<br>District: {district_filter}"
    if beat_filter != "ALL":
        area_note = f"<br>Beat: {beat_filter}"

    legend_html = f"""
    <div style="
        position: fixed; bottom: 30px; left: 30px; z-index: 1000;
        background: white; border: 1px solid #ccc; border-radius: 8px;
        padding: 14px 18px; font-family: Arial; font-size: 13px;
        box-shadow: 2px 2px 8px rgba(0,0,0,0.15); min-width: 240px;">
        <b style="font-size:14px; color:#1F3A6E;">🗺 Patrol Dispatch Map</b><br>
        <span style="color:#555; font-size:12px;">
            {query_date} &nbsp;|&nbsp; {shift_name}{area_note}
        </span>
        <hr style="margin:8px 0; border-color:#eee;">
        <table style="width:100%; border-collapse:collapse;">
          <tr><td><span style="background:#E74C3C;border-radius:3px;padding:2px 8px;">&nbsp;</span></td>
              <td style="padding-left:8px;">Critical (&gt;35%)</td>
              <td style="text-align:right;font-weight:bold;">{n_critical}</td></tr>
          <tr><td><span style="background:#E8933C;border-radius:3px;padding:2px 8px;">&nbsp;</span></td>
              <td style="padding-left:8px;">High (20–35%)</td>
              <td style="text-align:right;font-weight:bold;">{n_high}</td></tr>
          <tr><td><span style="background:#27AE60;border-radius:3px;padding:2px 8px;">&nbsp;</span></td>
              <td style="padding-left:8px;">Flagged total</td>
              <td style="text-align:right;font-weight:bold;">{n_flagged}</td></tr>
          <tr><td><span style="background:#F39C12;opacity:0.5;border-radius:3px;padding:2px 8px;">&nbsp;</span></td>
              <td style="padding-left:8px;">Monitor</td>
              <td style="text-align:right;font-weight:bold;">{n_monitor}</td></tr>
          <tr><td><span style="border:2px dashed #1A237E;border-radius:3px;padding:2px 8px;">&nbsp;</span></td>
              <td style="padding-left:8px;">Beat boundaries</td><td></td></tr>
          <tr><td><span style="border:2px dashed #6A1B9A;border-radius:3px;padding:2px 8px;">&nbsp;</span></td>
              <td style="padding-left:8px;">Community areas</td><td></td></tr>
        </table>
        <hr style="margin:8px 0; border-color:#eee;">
        <span style="color:#888; font-size:11px;">
            Threshold: {threshold:.0%} &nbsp;|&nbsp;
            Tiles: {n_flagged}/{n_total} ({n_flagged/max(n_total,1):.1%})
        </span>
    </div>"""
    m.get_root().html.add_child(folium.Element(legend_html))

    # Zoom to selected area if filtering
    if district_filter != "ALL" or beat_filter != "ALL":
        if len(display_df) > 0:
            lats = display_df["h3_address"].apply(lambda h: h3.cell_to_latlng(h)[0])
            lons = display_df["h3_address"].apply(lambda h: h3.cell_to_latlng(h)[1])
            m.fit_bounds([[lats.min(), lons.min()], [lats.max(), lons.max()]])

    return m


# =============================================================================
# INTERACTIVE WIDGET — map view with all new controls
# =============================================================================
date_w = widgets.DatePicker(
    description="Date:", value=pd.Timestamp.today().date(),
    layout=widgets.Layout(width="260px"))
shift_w = widgets.Dropdown(
    options=[("Morning / Noon  (06:00–13:59)", "morning_noon"),
             ("Afternoon / Night (14:00–21:59)", "afternoon_night"),
             ("Overnight  (22:00–05:59)", "overnight")],
    value="afternoon_night", description="Shift:",
    layout=widgets.Layout(width="380px"))
show_all_w = widgets.Checkbox(
    value=False, description="Show monitor layer (below threshold)",
    layout=widgets.Layout(width="340px"))

# ★ NEW map controls (mirror STEP 2)
map_threshold_slider = widgets.FloatSlider(
    value=engine.threshold, min=0.05, max=0.50, step=0.01,
    description="Threshold:", readout_format=".2f",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "90px"})

map_district_dd = widgets.Dropdown(
    options=[("All districts", "ALL")] + [(f"District {d}", d) for d in _all_districts],
    value="ALL", description="District:",
    layout=widgets.Layout(width="280px"),
    style={"description_width": "90px"})

map_beat_dd = widgets.Dropdown(
    options=[("All beats", "ALL")] + [(f"Beat {b}", b) for b in _all_beats],
    value="ALL", description="Beat:",
    layout=widgets.Layout(width="280px"),
    style={"description_width": "90px"})

map_tier_filter = widgets.SelectMultiple(
    options=["Critical", "High", "Moderate", "Low"],
    value=["Critical", "High", "Moderate", "Low"],
    description="Show tiers:", layout=widgets.Layout(width="280px", height="100px"),
    style={"description_width": "90px"})

def _update_map_beats(change):
    dist = change["new"]
    if dist == "ALL":
        opts = [("All beats", "ALL")] + [(f"Beat {b}", b) for b in _all_beats]
    else:
        beats_in_dist = sorted({b for b, d in tile_beat_map.values()
                                if d == dist and b != "Unknown"})
        opts = [("All beats in district", "ALL")] + [(f"Beat {b}", b) for b in beats_in_dist]
    map_beat_dd.options = opts
    map_beat_dd.value   = "ALL"

map_district_dd.observe(_update_map_beats, names="value")

map_live_btn = widgets.Button(
    description="🔄 Fetch 2026 & Retrain", button_style="warning",
    layout=widgets.Layout(width="200px", height="36px"))
map_live_status = widgets.Label(value="Live data: not loaded")

def on_map_live_refresh(b):
    global engine
    map_live_status.value = "Fetching 2026 data & checking for updates …"
    try:
        # 1) Retrain if there's new 2026 data
        engine, was_retrained = retrain_with_live_data(engine)

        # 2) Fetch fresh lag overrides
        qdate = date_w.value or pd.Timestamp.today().date()
        override_df = engine.live_override_from_api(target_date=qdate)
        _live_override_cache["df"] = override_df
        n = len(override_df) if override_df is not None else 0
        tag = "retrained + " if was_retrained else ""
        _live_override_cache["label"] = f"✓ {tag}{n} tiles with live lag ({qdate})"
        map_live_status.value = _live_override_cache["label"]
        try:
            live_status.value = _live_override_cache["label"]
        except Exception:
            pass
    except Exception as ex:
        map_live_status.value = f"✗ Error: {ex}"

map_live_btn.on_click(on_map_live_refresh)

run_btn = widgets.Button(
    description="🗺  Generate Map", button_style="success",
    layout=widgets.Layout(width="200px", height="36px"))
map_out = widgets.Output()

def on_map_run(b):
    with map_out:
        clear_output(wait=True)

        qdate     = date_w.value
        shift     = shift_w.value
        threshold = map_threshold_slider.value
        district  = map_district_dd.value
        beat      = map_beat_dd.value
        tiers     = list(map_tier_filter.value)

        if qdate is None:
            print("Please select a date.")
            return

        override = _live_override_cache["df"]
        override_tag = " [🔄 live]" if override is not None else ""

        print(f"Scoring all tiles for {qdate} | {shift} "
              f"| threshold {threshold:.2f}{override_tag} …")

        all_results = engine.predict(
            query_date=qdate, shift=shift,
            override_tiles=override, return_all=True,
            threshold_override=threshold)

        n_flagged = int(all_results['flagged'].sum())
        print(f"  {n_flagged} tiles flagged for dispatch "
              f"({n_flagged/len(all_results):.1%} of all tiles)")
        print(f"  Building map …")

        patrol_map = build_patrol_map(
            results=all_results, beats_gdf=beats_gdf,
            threshold=threshold, show_all_tiles=show_all_w.value,
            district_filter=district, beat_filter=beat,
            tier_filter_list=tiers,
            community_gdf=community_gdf)

        map_path = f"patrol_map_{qdate}_{shift}.html"
        patrol_map.save(map_path)
        print(f"  ✓ Map saved → {map_path}")
        print(f"  Open the file in a browser for full interactivity.\n")
        display(patrol_map)

run_btn.on_click(on_map_run)

print("╔═══════════════════════════════════════════════════════════════╗")
print("║    PATROL DISPATCH MAP — Tiles + Beats + Community Areas      ║")
print("║    🔄 Live Data  |  🎚 Threshold  |  📍 District/Beat Filter  ║")
print("╚═══════════════════════════════════════════════════════════════╝\n")

display(widgets.VBox([
    widgets.HBox([date_w, shift_w]),
    widgets.HBox([map_threshold_slider, show_all_w]),
    widgets.HBox([map_district_dd, map_beat_dd, map_tier_filter]),
    widgets.HBox([map_live_btn, map_live_status]),
    run_btn,
    map_out
]))


Loading Chicago Community Area boundaries …
  ✓ Loaded 77 community areas
╔═══════════════════════════════════════════════════════════════╗
║    PATROL DISPATCH MAP — Tiles + Beats + Community Areas      ║
║    🔄 Live Data  |  🎚 Threshold  |  📍 District/Beat Filter  ║
╚═══════════════════════════════════════════════════════════════╝

